<a href="https://colab.research.google.com/github/Sai-charan822/DSA03-NLP/blob/main/LAB/EXP19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 19. Word Sense Disambiguation using the Lesk Algorithm
We will implement the Lesk algorithm to resolve the correct meaning of an ambiguous word in a given context sentence by calculating the word overlap between the context and the dictionary definitions (glosses) of the word's senses.

In [9]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize

# Ensure required resources are downloaded, including punkt_tab
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

# A simple custom Lesk implementation for educational purposes
def simple_lesk(context_sentence, ambiguous_word):
    context = set(word_tokenize(context_sentence.lower()))
    # Remove stopwords to improve overlap accuracy
    stopwords = set(nltk.corpus.stopwords.words('english'))
    context = context - stopwords

    best_sense = None
    max_overlap = -1

    # Iterate through all synsets of the ambiguous word
    for sense in wn.synsets(ambiguous_word):
        # Combine definition and examples to create the signature/gloss
        signature = set(word_tokenize(sense.definition().lower()))
        for example in sense.examples():
            signature.update(word_tokenize(example.lower()))

        # Remove stopwords from the signature
        signature = signature - stopwords

        # Calculate overlap (intersection of context and signature)
        overlap = len(context.intersection(signature))

        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = sense

    return best_sense, max_overlap

# Test scenarios
scenarios = [
    ("The bank will charge interest on the loan.", "bank"),
    ("We went for a walk along the river bank.", "bank"),
    ("The biological researchers studied the division of the cell.", "cell")
]

print("Word Sense Disambiguation using Lesk:\n")
for sentence, word in scenarios:
    # Using our custom simple Lesk algorithm
    custom_sense, custom_score = simple_lesk(sentence, word)

    # Using NLTK's built-in Lesk algorithm
    tokenized_sentence = word_tokenize(sentence)
    nltk_sense = lesk(tokenized_sentence, word)

    print(f"Sentence: \"{sentence}\"")
    print(f"Target Word: '{word}'")
    if custom_sense:
        print(f"  -> Custom Lesk predicted sense: {custom_sense.name()} (Overlap score: {custom_score})")
        print(f"     Definition: {custom_sense.definition()}")
    if nltk_sense:
        print(f"  -> NLTK Lesk predicted sense  : {nltk_sense.name()}")
        print(f"     Definition: {nltk_sense.definition()}")
    print("-" * 80)

Word Sense Disambiguation using Lesk:

Sentence: "The bank will charge interest on the loan."
Target Word: 'bank'
  -> Custom Lesk predicted sense: bank.n.01 (Overlap score: 1)
     Definition: sloping land (especially the slope beside a body of water)
  -> NLTK Lesk predicted sense  : savings_bank.n.02
     Definition: a container (usually with a slot in the top) for keeping money at home
--------------------------------------------------------------------------------
Sentence: "We went for a walk along the river bank."
Target Word: 'bank'
  -> Custom Lesk predicted sense: bank.n.01 (Overlap score: 2)
     Definition: sloping land (especially the slope beside a body of water)
  -> NLTK Lesk predicted sense  : savings_bank.n.02
     Definition: a container (usually with a slot in the top) for keeping money at home
--------------------------------------------------------------------------------
Sentence: "The biological researchers studied the division of the cell."
Target Word: 'cell'
